# Exercise 03: Adaptive Attacks Against Defenses

**Goal:** Break a noise detector (Part A) and a randomised preprocessing defense (Part B) by crafting attacks specifically designed against each.

**Time:** ~10 min

## Background

A *non-adaptive* attack ignores the defense mechanism. An *adaptive* attack incorporates the defense into the attack objective.

**Part A — Detector evasion:** add a smoothness penalty to the PGD loss so the perturbation doesn't look noisy:

$$\mathcal{L}_\text{total} = \mathcal{L}_\text{clf} + \lambda \cdot \mathcal{L}_\text{det}$$

**Part B — EoT (Expectation over Transformations, Athalye 2018):** average gradients over $K$ random transformations to produce perturbations robust to randomised preprocessing:

$$\nabla_x \mathcal{L}_\text{EoT} = \frac{1}{K} \sum_{k=1}^{K} \nabla_x \mathcal{L}(\theta, t_k(x), y)$$

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

model = torch.hub.load('chenyaofo/pytorch-cifar-models', 'cifar10_resnet20', pretrained=True)
model.eval()

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616])
])
testset = torchvision.datasets.CIFAR10('../data', train=False, download=True, transform=transform)
images, labels = next(iter(torch.utils.data.DataLoader(testset, batch_size=50, shuffle=False)))
targets = (labels + 3) % 10
IMG_MIN, IMG_MAX = images.min().item(), images.max().item()

# Store clean images for perturbation-based detection
clean_images = images.clone()

# ── Simple heuristic detector ──
# Measures high-frequency content of the PERTURBATION (x - clean_images).
# Clean images score 0; adversarial perturbations score > 0 if noisy.
def high_freq_power(x):
    """High-frequency power of the perturbation relative to the clean image."""
    delta = x - clean_images
    kernel = (torch.ones(1,1,3,3)/9).expand(3,-1,-1,-1)
    smoothed = nn.functional.conv2d(delta, kernel, padding=1, groups=3)
    return ((delta - smoothed)**2).mean(dim=[1,2,3])

def detect(x, threshold=0.001):
    """Returns 1 (adversarial) or 0 (clean) per image."""
    return (high_freq_power(x) > threshold).float()

def eval_attack(x_adv, true_labels, targets, label):
    acc = (model(x_adv).argmax(1) == true_labels).float().mean()
    tgt = (model(x_adv).argmax(1) == targets).float().mean()
    det = detect(x_adv).mean()
    print(f"{label:35s}  clean-acc={acc:.0%}  target-hit={tgt:.0%}  detection-rate={det:.0%}")

# Baseline standard targeted PGD
def pgd_targeted_std(images, targets, epsilon=0.05, alpha=0.005, steps=40):
    x = images.clone().detach()
    x = x + torch.empty_like(x).uniform_(-epsilon, epsilon)
    x = torch.clamp(x, IMG_MIN, IMG_MAX)
    for _ in range(steps):
        x = x.detach().requires_grad_(True)
        loss = nn.CrossEntropyLoss()(model(x), targets)
        loss.backward()
        with torch.no_grad():
            x = x - alpha * x.grad.sign()
            delta = torch.clamp(x - images, -epsilon, epsilon)
            x = torch.clamp(images + delta, IMG_MIN, IMG_MAX)
    return x.detach()

x_std = pgd_targeted_std(images, targets)
eval_attack(x_std, labels, targets, "Standard targeted PGD")
print()
print(f"Clean image power:  {high_freq_power(images).mean().item():.6f}   (always 0 — no perturbation)")
print(f"Standard PGD power: {high_freq_power(x_std).mean().item():.6f}   (noisy perturbation → detected)")
print()
print("Notice: standard PGD has a high detection rate because")
print("it creates high-frequency perturbation patterns.")

## 🎯 Part A: Evading a Noise Detector (TODO)

**Hint:** $\mathcal{L}_\text{total} = \mathcal{L}_\text{clf} + \lambda \cdot \texttt{high\_freq\_power}(x)\text{.mean()}$

Minimising $\mathcal{L}_\text{total}$ pushes toward the target class (minimize $\mathcal{L}_\text{clf}$) while suppressing high-frequency noise (minimize $\mathcal{L}_\text{det}$). Use $\lambda \approx 500$ to balance the two terms — $\mathcal{L}_\text{clf}$ is on the order of ~2, while $\mathcal{L}_\text{det}$ is on the order of ~0.001.

In [ ]:
def pgd_evasion_attack(images, targets, epsilon=0.05, alpha=0.005, steps=80, lam=500.0):
    """
    Adaptive PGD that simultaneously:
      (A) fools the classifier (targeted: reach `targets`)
      (B) evades the noise detector (minimise high-frequency energy of the perturbation)

    TODO — implement the joint-loss attack:
      At each step compute:
        # TODO: L_clf   = CrossEntropyLoss(model(x), targets)
        # TODO: L_det   = high_freq_power(x).mean()
        # TODO: L_total = L_clf + lam * L_det
      Then: gradient step (descent: x = x - alpha * grad.sign())
            project back onto epsilon-ball
            clip to valid range

    Note: lam ≈ 500 balances L_clf (~2.0) against L_det (~0.001).
    Hint: `high_freq_power` returns a tensor of shape (N,); call .mean() to get a scalar.
    """
    x = images.clone().detach()
    x = x + torch.empty_like(x).uniform_(-epsilon, epsilon)
    x = torch.clamp(x, IMG_MIN, IMG_MAX)
    # 🎯 TODO: implement the loop
    return x.detach()

x_evasion = pgd_evasion_attack(images, targets)
eval_attack(x_evasion, labels, targets, "Evasion attack (joint loss)")

## 🎯 Part B: Evading Randomised Preprocessing with EoT (TODO)

**Hint:** Accumulate gradients over $K$ random transforms inside the loop, then average them before taking the step:
$$\text{avg\_grad} = \frac{1}{K}\sum_{k=1}^{K} \nabla_x \mathcal{L}(\theta,\, t_k(x),\, y)$$

In [ ]:
def random_preprocess(x):
    """Randomised preprocessing defense: strong brightness jitter + Gaussian noise."""
    factor = float(np.random.uniform(0.70, 1.30))   # ±30% brightness
    noise  = torch.randn_like(x) * 0.10              # σ=0.10 noise
    return torch.clamp(x * factor + noise, IMG_MIN, IMG_MAX)

# Show that standard PGD fails against randomized preprocessing
def eval_random_defense(x_adv, targets, n_trials=30):
    """Evaluate adversarial success rate under random preprocessing (average over trials)."""
    hits = 0
    with torch.no_grad():
        for _ in range(n_trials):
            hits += (model(random_preprocess(x_adv)).argmax(1) == targets).float().sum().item()
    return hits / (n_trials * len(targets))

with torch.no_grad():
    hit_no_def = (model(x_std).argmax(1) == targets).float().mean().item()
print(f"Standard PGD — no defense:                    target-hit={hit_no_def:.0%}")
print(f"Standard PGD — under random defense:          target-hit={eval_random_defense(x_std, targets):.0%}")
print()

def pgd_eot_attack(images, targets, epsilon=0.05, alpha=0.005, steps=60, K=16):
    """
    EoT (Expectation over Transformations) attack for randomized preprocessing.

    TODO — implement EoT:
      For each step:
        1. Accumulate gradients over K random transformations:
             For k in range(K):
               x_t = random_preprocess(x)  <- apply random transform
               # TODO: compute grad of CrossEntropyLoss(model(x_t), targets) w.r.t. x
               # TODO: accumulate grad
        2. # TODO: avg_grad = accumulated_grad / K
        3. # TODO: x = x - alpha * avg_grad.sign()   (targeted: MINUS sign)
        4. project back, clip

    Hint: use x.detach().requires_grad_(True) at the start of each step,
          then accumulate .grad after each backward() call, setting it to None between k iterations.
    """
    x = images.clone().detach()
    x = x + torch.empty_like(x).uniform_(-epsilon, epsilon)
    x = torch.clamp(x, IMG_MIN, IMG_MAX)
    # 🎯 TODO: implement EoT loop
    return x.detach()

x_eot = pgd_eot_attack(images, targets)
print(f"EoT PGD   — under random defense:             target-hit={eval_random_defense(x_eot, targets):.0%}")

## Discussion

In [ ]:
print("""
=== Discussion ===

1. Why does standard PGD have a high detection rate?
   Standard PGD optimises only the classification loss; the resulting perturbations have
   high-frequency checkerboard patterns that are easily flagged by a noise detector.

2. Why does the joint-loss attack evade the detector?
   The added L_det term penalises high-frequency energy, so the optimiser is forced
   to spread perturbations more smoothly — sacrificing some attack power for stealthiness.

3. Why does EoT succeed against randomised preprocessing?
   Standard PGD optimises for ONE realisation of the preprocessing. Under a different
   realisation the perturbation lands in a different place and the attack fails.
   EoT averages gradients over K realisations, so the update direction is robust
   to the randomness — the perturbation 'works on average'.

4. What does this imply about defense papers?
   A defense should always be evaluated against an adaptive adversary that knows and
   optimises against the defense mechanism. Non-adaptive evaluation systematically
   overestimates robustness (Carlini et al., 2019).
""")
